# Caching layers & cache-aware routing: three caches, and making them hit

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 40 §40.2, §40.4 · type: concept-lab*

**The promise:** by the end you can build each of the three cache layers, measure its hit rate and savings, know which to reach for first, meter the retrieval cost plane, and route so the cache you paid to build actually hits.

## 🧠 Why this matters

The cheapest, fastest token is the one you never generate. Caching for LLM systems comes in **three distinct layers**, and conflating them is the classic source of confusion:

- **Exact cache** — return a stored response when the *identical* request arrives again. Safe by construction; collapses when users phrase freely.
- **Semantic cache** — serve a stored response when a *similar enough* past query exists. Catches paraphrases; risky at the threshold.
- **Provider prompt cache** — the provider reuses computation on a stable **prefix** of your prompt. Near-free, huge for agents; a strict prefix match.

This notebook builds all three offline with a mock model and a seeded local embedder, so every hit-rate and savings number is computed deterministically. See §40.2 and §40.4.

## Objectives & prereqs

**By the end you can:**
- Build the book's `cache_key` exact cache and measure its hit rate on a repeating workload.
- Build a semantic cache on a seeded offline embedder, and feel why the threshold is dangerous.
- Simulate provider prompt-cache prefix reuse — and catch the **timestamp killer** that silently zeroes the hit rate.
- Meter the **retrieval cost plane** (embedding / reranker / index) that the generation meter never sees.
- Route **prefix-aware** so same-prefix requests reuse a KV-cache instead of each paying full prefill.

**Prereqs:** `40-01-token-accounting-and-attribution`; Ch 11 (cache-aside) and Ch 13 (vector similarity) read.

**Runs free & offline.** Mock model + a tiny **seeded** offline embedder; the query set is committed in `data/queries.csv` with deliberate exact-repeats and paraphrase-pairs. No API key, no network.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import csv
import json
import math
import random
import hashlib

from dotenv import load_dotenv

load_dotenv()

# MOCK=1 (default): offline mock model + deterministic local embedder. The cache logic
# is identical against a live provider; mocking just makes hit rates reproducible.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(40)  # the offline embedder is seeded so cosine similarity is deterministic

DATA = os.path.join(os.getcwd(), "data")
print("MOCK mode:", MOCK, "— offline, no API key needed" if MOCK else "— LIVE (would call the SDK)")

## A mock model that *counts* what it generated

To measure savings we need a model call that records what it cost. This stand-in returns a canned answer and tracks generated output tokens, so "tokens saved by the cache" is a real, computed number rather than a claim.

In [ ]:
GEN_LOG = {"calls": 0, "output_tokens": 0}


def mock_generate(prompt):
    """A canned 'model' that meters itself. Deterministic; no network."""
    GEN_LOG["calls"] += 1
    GEN_LOG["output_tokens"] += 60  # pretend each fresh answer is ~60 output tokens
    return f"[answer to: {prompt[:48]}...]"


def reset_gen():
    GEN_LOG["calls"] = 0
    GEN_LOG["output_tokens"] = 0

## Layer 1 — exact cache (`cache_key`)

The book's exact cache is the cache-aside pattern from Ch 11, with the key derived by **hashing the full request** (model + messages + params). A hit is, by construction, the right answer. We replay a repeating workload and watch the hit rate.

In [ ]:
def cache_key(model, messages, **params):
    payload = json.dumps(
        {"model": model, "messages": messages, "params": params},
        sort_keys=True,
    )
    return "llm:" + hashlib.sha256(payload.encode()).hexdigest()


EXACT_CACHE = {}


def cached_call(model, messages, **params):
    key = cache_key(model, messages, **params)
    if (hit := EXACT_CACHE.get(key)) is not None:
        return hit, True            # cache hit
    text = mock_generate(messages[-1]["content"])
    EXACT_CACHE[key] = text
    return text, False              # miss -> generated


def load_queries(path):
    with open(path, newline="", encoding="utf-8") as f:
        return [(row["tenant"], row["query"]) for row in csv.DictReader(f)]


queries = load_queries(os.path.join(DATA, "queries.csv"))
reset_gen()
EXACT_CACHE.clear()

hits = 0
for tenant, q in queries:
    # NOTE: scope the key per tenant so one customer's answer can't leak to another.
    msgs = [{"role": "user", "content": q}]
    _, was_hit = cached_call("claude-haiku-4-5", msgs, tenant=tenant)
    hits += was_hit

print(f"{len(queries)} queries, exact-cache hits: {hits}  "
      f"({hits / len(queries) * 100:.0f}% hit rate)")
print(f"model calls actually made: {GEN_LOG['calls']}  "
      f"(saved {len(queries) - GEN_LOG['calls']} generations)")

**What you just saw.** The exact cache only fires on *byte-identical* repeats — "how do I reset my password" hits, but "password reset steps" is a complete miss even though it wants the same answer. Safe, but its hit rate collapses the moment users phrase things freely. That gap is exactly what the next layer attacks.

## Layer 2 — semantic cache (and its dangerous threshold)

A semantic cache embeds the incoming query and serves a stored answer when a *similar enough* past query exists (cosine similarity above a threshold, the vector machinery from Ch 13). It catches paraphrases. Its danger is the threshold — and a real trap lives in our query set: **"fee for plan A"** and **"fee for plan B"** sit a hair apart in embedding space but have *opposite* answers.

Our embedder is a tiny **seeded, offline** hashing embedder — not a real model, but deterministic and good enough to *feel* the threshold trade-off without a network call.

In [ ]:
def embed(text, dim=64):
    """Deterministic offline embedder: hash tokens into a bag-of-words vector.

    Not a real semantic model — but stable and seeded, so similarity is reproducible
    and the threshold lesson is visible without any API call.
    """
    vec = [0.0] * dim
    for tok in text.lower().split():
        h = int(hashlib.sha256(tok.encode()).hexdigest(), 16)
        vec[h % dim] += 1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def cosine(a, b):
    return sum(x * y for x, y in zip(a, b))


class SemanticCache:
    def __init__(self, threshold):
        self.threshold = threshold
        self.store = {}   # tenant -> [(embedding, query, answer)]

    def get(self, tenant, query):
        qv = embed(query)
        best, best_sim = None, -1.0
        for ev, q, ans in self.store.get(tenant, []):   # per-tenant scope: never cross tenants
            sim = cosine(qv, ev)
            if sim > best_sim:
                best, best_sim = (q, ans), sim
        if best is not None and best_sim >= self.threshold:
            return best[1], best_sim
        return None, best_sim

    def put(self, tenant, query, answer):
        self.store.setdefault(tenant, []).append((embed(query), query, answer))

## 🔮 Predict: what does a too-loose threshold do?

We'll run the same queries through a semantic cache at a **loose** threshold (`0.80`) and a **conservative** one (`0.95`). The query set deliberately contains the `plan A` / `plan B` pair.

**Predict, before running:** at the loose threshold, will "what is the fee for plan B" be served the *cached "plan A" answer*? And which threshold gives the higher hit rate — and is higher better here? Write your guess, then run it.

In [ ]:
def run_semantic(threshold):
    reset_gen()
    cache = SemanticCache(threshold)
    hits, wrong = 0, 0
    for tenant, q in queries:
        cached, sim = cache.get(tenant, q)
        if cached is not None:
            hits += 1
            # Detect a dangerous near-miss: a 'plan B' query served the 'plan A' answer.
            if "plan b" in q.lower() and "plan a" in cached.lower():
                wrong += 1
        else:
            cache.put(tenant, q, mock_generate(q))
    return hits, wrong, GEN_LOG["calls"]


for thr in (0.80, 0.95):
    hits, wrong, calls = run_semantic(thr)
    print(f"threshold {thr:.2f}: {hits:>2d} hits, {wrong} WRONG-answer hits, "
          f"{calls} model calls")

**What you just saw.** The loose threshold serves more "hits" — including, dangerously, the **plan A answer to a plan B question**. That near-miss reads to the user as a bug, not a cache. The conservative threshold gives up some hit rate to stay correct. The semantic cache is the one layer where a higher hit rate can be *worse*. Keep the threshold conservative, scope keys **per tenant** (done above), and use it only where near-misses are tolerable.

## Layer 3 — provider prompt cache (prefix reuse), and the killer

The provider reuses computation for a *prefix* of your prompt — system prompt, tool definitions, shared context — charging cached tokens at a fraction of input price. It is a **strict prefix match**: order stable-first (frozen system prompt, deterministic tools), keep volatile content (timestamps, the user's question) last. A single early-byte change invalidates everything after it.

We simulate prefix-match savings, then reproduce the classic killer: a **timestamp in the system prompt** silently drops the hit rate to zero — *no error, just full price*.

In [ ]:
def shared_prefix_len(a, b):
    """Length of the common leading substring — the provider's cacheable prefix."""
    n = 0
    for ca, cb in zip(a, b):
        if ca != cb:
            break
        n += 1
    return n


CACHE_READ_RATE = 0.10  # cached prefix tokens cost ~1/10th of fresh input here


def prefill_cost(prompt, previous):
    """Tokens billed at full price vs. read from the prompt cache. ~4 chars/token."""
    if previous is None:
        return len(prompt) / 4, 0.0
    reuse = shared_prefix_len(prompt, previous)
    fresh = (len(prompt) - reuse) / 4
    cached = reuse / 4
    return fresh, cached


STATIC_SYSTEM = (
    "You are a support agent. Follow policy. Tools: lookup_order, issue_refund. "
)
# GOOD: volatile user question appended AFTER a frozen prefix.
good_prompts = [STATIC_SYSTEM + "User: where is my order?",
                STATIC_SYSTEM + "User: how do I get a refund?"]
fresh_g, cached_g = prefill_cost(good_prompts[1], good_prompts[0])

# BAD: a timestamp interpolated into the SYSTEM prompt changes the very first bytes.
bad_prompts = [f"Current time: 2026-06-20 14:32:07. {STATIC_SYSTEM}User: where is my order?",
               f"Current time: 2026-06-20 14:32:09. {STATIC_SYSTEM}User: how do I get a refund?"]
fresh_b, cached_b = prefill_cost(bad_prompts[1], bad_prompts[0])

print(f"stable-prefix prompt : fresh={fresh_g:5.1f} tok  cached={cached_g:5.1f} tok  "
      f"(cache_read>0  -> WORKING)")
print(f"timestamped prompt   : fresh={fresh_b:5.1f} tok  cached={cached_b:5.1f} tok  "
      f"(cache_read==0 -> KILLED)")

## ⚠️ Pitfall: the prompt-cache killer

A timestamp interpolated into the system prompt — `Current time: 2026-06-20 14:32:07` — changes every request and silently reduces your hit rate to **zero**. No error, no warning, just full price on every call. The diagnosis is mechanical: when cached-token counts read zero on traffic that *should* repeat, **diff two rendered prompts byte by byte** — the invalidator is in the first place they differ. And always **verify** via `cache_read_input_tokens` on the response; never assume the cache is working.

In [ ]:
idx = shared_prefix_len(bad_prompts[0], bad_prompts[1])  # first differing index
print("byte-diff of the two timestamped prompts:")
print(f"  first difference at index {idx}: "
      f"{bad_prompts[0][idx - 3:idx + 3]!r} vs {bad_prompts[1][idx - 3:idx + 3]!r}")
print("  -> the volatile timestamp sits BEFORE the static system prompt, so nothing caches.")
print("\nFix: move volatile content to the END; verify cache_read_input_tokens > 0 in prod.")

## Prompt compression — shrink what you send

Compression is a family of habits, not one trick: trim conversation history by summarizing old turns, prune verbose tool results to the fields the model needs, and dedupe retrieved chunks before stuffing them into context. Tokens you don't send are also tokens the model doesn't have to *read* — fewer tokens means lower latency and often **better** attention.

In [ ]:
def compress_context(history, tool_results, chunks):
    """Three compression habits, measured. Returns (before_tokens, after_tokens)."""
    def toks(s):
        return len(s) / 4

    before = (sum(toks(h) for h in history)
              + sum(toks(json.dumps(t)) for t in tool_results)
              + sum(toks(c) for c in chunks))

    # 1) Summarize old turns: replace all but the last 2 with a short summary.
    summarized = ["(summary of earlier turns)"] + history[-2:] if len(history) > 2 else history
    # 2) Prune tool results to needed fields only.
    pruned = [{"id": t["id"], "status": t["status"]} for t in tool_results]
    # 3) Dedupe retrieved chunks.
    deduped = list(dict.fromkeys(chunks))

    after = (sum(toks(h) for h in summarized)
             + sum(toks(json.dumps(t)) for t in pruned)
             + sum(toks(c) for c in deduped))
    return before, after


history = [f"turn {i}: some back-and-forth text that piles up over a long agent run" for i in range(8)]
tool_results = [{"id": i, "status": "ok", "raw": "x" * 400, "debug": "y" * 200} for i in range(3)]
chunks = ["retrieved passage about refunds"] * 4 + ["a distinct passage about shipping"]

before, after = compress_context(history, tool_results, chunks)
print(f"context before compression: {before:7.0f} tokens")
print(f"context after  compression: {after:7.0f} tokens  "
      f"({(1 - after / before) * 100:.0f}% smaller -> cheaper AND faster)")

## The retrieval cost plane — line items the generation meter never sees

A retrieval pipeline spends money the completion meter is blind to. Three line items hide there: **embedding-API calls** (every doc indexed and every query embedded), **reranker calls** (a cross-encoder scores every candidate for every query), and **vector-store infrastructure** (RAM/disk you rent whether or not anyone queries). For a retrieval-heavy agent these can *rival* generation cost — so meter them with the **same** per-feature, per-tenant labels.

In [ ]:
# Illustrative unit prices for the retrieval plane (config, not hardcoded in logic).
RETRIEVAL_PRICES = {
    "embed_per_1k_tokens": 0.0001,
    "rerank_per_candidate": 0.00005,
    "index_hourly_usd": 0.04,
}


def retrieval_cost(n_queries, tokens_per_query, candidates_per_query, hours=1):
    embed_cost = n_queries * tokens_per_query / 1000 * RETRIEVAL_PRICES["embed_per_1k_tokens"]
    rerank = n_queries * candidates_per_query * RETRIEVAL_PRICES["rerank_per_candidate"]
    infra = hours * RETRIEVAL_PRICES["index_hourly_usd"]
    return {"embedding": embed_cost, "reranker": rerank, "index_infra": infra}


# Naive: re-embed every query, rerank the top 100 candidates reflexively.
naive = retrieval_cost(n_queries=10_000, tokens_per_query=40, candidates_per_query=100)
# Optimized: cache query embeddings (50% recur), right-size the candidate set to 20.
opt = retrieval_cost(n_queries=5_000, tokens_per_query=40, candidates_per_query=20)

print("retrieval cost plane (per hour at 10k queries):")
for k in naive:
    print(f"  {k:12s} naive ${naive[k]:7.4f}   optimized ${opt[k]:7.4f}")
print(f"  {'TOTAL':12s} naive ${sum(naive.values()):7.4f}   "
      f"optimized ${sum(opt.values()):7.4f}")
print("\nCache query embeddings, batch them, right-size the reranker/candidate set.")

## Cache-aware (prefix-affinity) routing

The provider's prompt cache and the gateway's KV-cache both key off a **shared prompt prefix** — but a cache only pays off if requests sharing that prefix land where the cache lives. Round-robin **scatters** same-prefix requests across replicas, so each pays full prefill. The lever is the **routing key**: hash on the stable prefix (tenant / agent / prompt-version) so the second request reuses the first's KV-cache.

In [ ]:
N_REPLICAS = 4


def route_round_robin(requests):
    return [i % N_REPLICAS for i, _ in enumerate(requests)]


def route_prefix_aware(requests):
    # Hash the STABLE prefix (here: tenant + agent), not the volatile question.
    return [int(hashlib.sha256(f"{t}/{agent}".encode()).hexdigest(), 16) % N_REPLICAS
            for (t, agent, _q) in requests]


def kv_cache_hits(requests, routing):
    """A replica reuses its KV-cache when it sees the same prefix it saw before."""
    seen_per_replica = {}
    hits = 0
    for (t, agent, _q), replica in zip(requests, routing):
        prefix = f"{t}/{agent}"
        bucket = seen_per_replica.setdefault(replica, set())
        if prefix in bucket:
            hits += 1
        else:
            bucket.add(prefix)
    return hits


# 12 requests from 2 tenants/agents — lots of shared prefixes to exploit.
reqs = ([("acme", "support_agent", f"q{i}") for i in range(6)]
        + [("globex", "faq_bot", f"q{i}") for i in range(6)])

rr = route_round_robin(reqs)
pa = route_prefix_aware(reqs)
print(f"round-robin routing : KV-cache hits = {kv_cache_hits(reqs, rr):>2d} / {len(reqs)}")
print(f"prefix-aware routing: KV-cache hits = {kv_cache_hits(reqs, pa):>2d} / {len(reqs)}")
print("\nSame requests, same caches — only the routing KEY changed.")

**What you just saw.** Identical traffic, identical cache machinery — the *only* change is the routing key, and the prefix-aware router lands same-prefix requests on the same replica so the KV-cache actually hits. A cache you paid to build is worthless if the load balancer keeps moving the request away from it.

## 🎯 Senior lens

Apply optimizations **cheapest-first**: right-size the model → prompt caching → exact caching → batch API → *then* semantic caching and aggressive compression, which carry quality risk. Most teams reach for the risky end first because it sounds sophisticated; run the list top-down and you usually cut spend severalfold before touching anything that can degrade answers. Two more senior instincts from §40.4: put **embedding and reranker calls through the same metered chokepoint** as completions — a RAG agent optimized only on generation tokens still bleeds through an unmetered embedding pipeline — and make the router **cache-aware**, because a cache that never hits is just latency you paid for.

## Recap

- Three cache layers answer different questions: **exact** (identical request), **semantic** (similar request), **provider prompt** (shared prefix). Don't conflate them.
- Exact cache is safe but collapses on free phrasing; semantic cache catches paraphrases but its **threshold** can serve wrong near-misses — keep it conservative and **scope keys per tenant**.
- Provider prompt caching needs a **stable prefix**; a timestamp in the system prompt is the classic killer — diff prompts byte-by-byte and verify `cache_read_input_tokens > 0`.
- **Prompt compression** (summarize history, prune tool results, dedupe chunks) cuts cost *and* latency.
- The **retrieval cost plane** (embedding / reranker / index) can rival generation cost — meter it with the same labels and right-size it.
- **Prefix-aware routing** makes the cache you paid to build actually hit — the lever is the routing key.
- Order of leverage: **right-size model → prompt cache → exact cache → batch → semantic/compression.**

## Exercises

Each one *changes* something and asks you to predict first. (Solutions arrive in `solutions/` in Phase 2.)

1. **Tune the threshold.** Sweep the semantic threshold from `0.70` to `0.99` and print hit rate vs. wrong-answer count. Predict the threshold where wrong-answer hits first appear, then find it.
2. **Move the timestamp.** Rewrite `bad_prompts` to put the timestamp at the *end* (after the user question). Predict the new cached-token count, then compute it and confirm the cache revives.
3. **Right-size the reranker.** In `retrieval_cost`, find the `candidates_per_query` at which reranker cost equals embedding cost for 10k queries. Predict above/below 50 first.
4. **Break prefix routing.** Change `route_prefix_aware` to hash the *full request* (including the question). Predict the KV-cache hit count, then run it — and explain why it collapses to round-robin.

In [ ]:
# Exercise 1 — your code here.

In [ ]:
# Exercise 2 — your code here.

In [ ]:
# Exercise 3 — your code here.

In [ ]:
# Exercise 4 — your code here.

## Next

- **Next notebook:** [`40-03-latency-budgets-parallelism-and-speculation.ipynb`](40-03-latency-budgets-parallelism-and-speculation.ipynb) — cost is paid by the company, latency by the user; engineer it on purpose.
- **Book:** §40.2 (caching & compression), §40.4 (retrieval cost plane + prefix-aware routing), the token-optimization playbook table, and the §40 checklist; Ch 11 (cache-aside), Ch 13 (vector similarity).
- **Blueprint this feeds:** [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) (the three cache layers + cache-aware routing) and [`blueprints/rag-pipeline/`](../../../blueprints/rag-pipeline/) (the retrieval cost plane).
- **Capstone:** advances [`capstone/llm/gateway.py`](../../../capstone/llm/gateway.py) — exact + semantic + prompt caching and prefix-aware routing (`checkpoints/ch40-cost-and-caching`).